# OPES free energy simulation


OPES, On-the-fly Probability Enhanced Sampling, is an adaptive enhanced-sampling
method which introduce an external bias potential to favor the exploration of 
the relevant phase space, so configurations that would be rarely visited
in unbiased molecular dynamics become accessible on shorter timescales.
The bias is built as a function of selected collective variables, which represent a low
dimensional representation of the important, slow modes of the system.

In this tutorial we use OPES through PLUMED to sample the dissociation of N2 on
Fe(111). The physical forces are computed by a pretrained MACE-MP
machine-learning potential, while PLUMED adds a bias on two collective variables:
the N-N distance and the coordination between nitrogen and the Fe surface. The
notebook runs one short simulation; a Python script is also available.


## Computational ingredients

The workflow combines ASE for molecular dynamics, MACE for the interatomic
potential, and PLUMED for the enhanced-sampling bias.


In [4]:
import os
import time
from pathlib import Path

import numpy as np
import torch
from ase import units
from ase.calculators.plumed import Plumed
from ase.io import Trajectory, read, write
from ase.md.bussi import Bussi
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import mace_mp


## Choose one simulation

OPES comes with different variants: `OPES_METAD_EXPLORE` is useful for broad initial
exploration of the collective-variable space, while `OPES_METAD` is the standard
OPES metadynamics, which can be used for free energy convergence given a good set of CVs. 


In [5]:
temperature = 700.0 # K
bias = "OPES_METAD_EXPLORE"  # Alternative: "OPES_METAD"
barrier = 0.8       # eV

# MD settings.
timestep = 0.5      # fs
total_time = 1.0    # ps --> only for quick testing, increase for production runs
taut = 100.0        # fs
interval_info = 100
interval_traj = interval_info

device = "cuda:0" if torch.cuda.is_available() else "cpu"


## Prepare the run directory

Each run writes its PLUMED input, energy log, and trajectory files to a directory
named after the temperature, OPES mode, and barrier. This keeps exploratory
notebook tests separate from production runs submitted with the Python script.


In [6]:
label = "metad" if bias == "OPES_METAD" else "explore"
outdir = Path(f"opes_{int(temperature)}K_{label}_b{barrier}")
outdir.mkdir(parents=True, exist_ok=True)

os.chdir(outdir)
print(f"Writing simulation files in: {Path.cwd()}")


Writing simulation files in: /leonardo_scratch/fast/IscrB_ProAmmo/CompCatSchool/1_opes/opes_700K_explore_b0.8


## Use a pretrained MACE potential

We start from the Fe(111) + N2 structure prepared in `0_system` and attach a
pretrained MACE model. This provides a
machine-learning approximation to the potential-energy surface, avoiding any
model training during this first tutorial.


In [7]:
system_path = Path( "../../0_system/init_config.xyz" )

atoms = read(system_path)
calc = mace_mp(model="mh-0", head="oc20_usemppbe", device=device)

print(f"Loaded {len(atoms)} atoms from {system_path}")
print(f"Chemical formula: {atoms.get_chemical_formula()}")


Using Materials Project MACE for MACECalculator with /leonardo/home/userexternal/lbonati1/.cache/mace/macemh0model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Loaded 98 atoms from ../../0_system/init_config.xyz
Chemical formula: Fe96N2


/leonardo_scratch/fast/IscrB_ProAmmo/envs/compcatschool/lib/python3.12/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


## PLUMED input: CVs and bias definition

The OPES bias is applied to two physically interpretable collective variables:
`d_N2`, which follows stretching and dissociation of the molecule, and `c_N_Fe`,
which tracks interaction with the Fe surface. The upper walls prevent the molecule from desorbing
and running away, as well as facilitating the reversible sampling of the barrier by 
preventing to relax in a much more stable product state.

In [20]:
atomic_numbers = atoms.get_atomic_numbers()
fe_indices = np.flatnonzero(atomic_numbers == 26) + 1
n_indices = np.flatnonzero(atomic_numbers == 7) + 1

if len(n_indices) != 2:
    raise ValueError(f"Expected exactly two nitrogen atoms, found {len(n_indices)}")

plumed_text = f"""
UNITS LENGTH=A ENERGY=eV

Fe: GROUP ATOMS={','.join(map(str, fe_indices))}
N: GROUP ATOMS={','.join(map(str, n_indices))}

d_N2: DISTANCE ATOMS={n_indices[0]},{n_indices[1]}
com_N2: COM ATOMS=N
c_N_Fe: COORDINATION GROUPA=N GROUPB=Fe R_0=2.5

w_d_N2: UPPER_WALLS ARG=d_N2 AT=2 KAPPA=0.2 EXP=2 EPS=0.1
w_com_N2: UPPER_WALLS ARG=com_N2.z AT=10 KAPPA=1

opes: {bias} ARG=d_N2,c_N_Fe PACE=100 BARRIER={barrier} TEMP={temperature} STATE_WFILE=STATES STATE_WSTRIDE=100

PRINT STRIDE={interval_info} ARG=* FILE=COLVAR
FLUSH STRIDE=100
"""

Path("plumed.dat").write_text(plumed_text)
print(plumed_text)



UNITS LENGTH=A ENERGY=eV

Fe: GROUP ATOMS=1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96
N: GROUP ATOMS=97,98

d_N2: DISTANCE ATOMS=97,98
com_N2: COM ATOMS=N
c_N_Fe: COORDINATION GROUPA=N GROUPB=Fe R_0=2.5

w_d_N2: UPPER_WALLS ARG=d_N2 AT=2 KAPPA=0.2 EXP=2 EPS=0.1
w_com_N2: UPPER_WALLS ARG=com_N2.z AT=10 KAPPA=1

opes: OPES_METAD_EXPLORE ARG=d_N2,c_N_Fe PACE=100 BARRIER=0.8 TEMP=700.0 STATE_WFILE=STATES STATE_WSTRIDE=100

PRINT STRIDE=100 ARG=* FILE=COLVAR
FLUSH STRIDE=100



## Combine MACE and PLUMED

ASE wraps the MACE calculator with PLUMED. At every MD step, MACE evaluates the
underlying energy and forces, and PLUMED adds the OPES bias and writes the
collective-variable output needed for later analysis.


In [ ]:
plumed_input = Path("plumed.dat").read_text().splitlines()
plumed_calc = Plumed(calc, plumed_input, timestep * units.fs, atoms, units.kB * temperature)
atoms.calc = plumed_calc

## Initialize molecular dynamics

The trajectory is propagated with a Bussi thermostat at the selected temperature.
This gives us a controlled finite-temperature dynamics while OPES accelerates
sampling along the chosen variables.


In [23]:
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature)
dyn = Bussi(atoms, timestep * units.fs, temperature, taut * units.fs)


## Save diagnostics and configurations

The run produces a compact set of files: `ENERGY` for thermodynamic diagnostics,
`COLVAR`/`KERNELS`/`STATES` from PLUMED, and `traj.traj` for saved atomic
configurations.


In [24]:
t0 = time.time()
log_file = Path("ENERGY").open("w")
log_file.write("# time_ps Epot_eV Ekin_eV Etot_eV Temp_K CPU_Time_s\n")
log_file.flush()


def log_status(a=atoms, dyn=dyn, f=log_file):
    epot = a.get_potential_energy()
    if np.ndim(epot) > 0:
        epot = epot[0]
    epot = float(epot)

    ekin = float(a.get_kinetic_energy())
    etot = epot + ekin
    temp = float(a.get_temperature())
    time_ps = dyn.get_time() / units.fs / 1000
    cpu_time = time.time() - t0

    f.write(
        f"{time_ps:12.6f} {epot:16.8f} {ekin:16.8f} "
        f"{etot:16.8f} {temp:12.6f} {cpu_time:12.6f}\n"
    )
    f.flush()


dyn.attach(log_status, interval_info)
trajectory = Trajectory("traj.traj", "w", atoms)
dyn.attach(trajectory, interval_traj)


## Run MD simulation


In [ ]:
nsteps = int((total_time * 1000) // timestep)
print(f"Running {nsteps} MD steps ({total_time} ps at {timestep} fs/step)")

dyn.run(nsteps)

log_file.close()
trajectory.close()

## Export for visualization

ASE's trajectory format is convenient for Python analysis. The XYZ conversion is
optional, but useful for inspecting the biased trajectory in standard molecular
viewers.


In [ ]:
frames = read("traj.traj", index=":")
write("traj.xyz", frames)

print(f"Wrote {len(frames)} frames to {Path.cwd() / 'traj.xyz'}")

## Analysis (WIP)